# Closed-form equivalent layer associated with gravitational field

This code illustrates how to compute a planar closed-form equivalent layer that reproduces the gravitational potential and its derivatives at points above the layer.

### Things we have to import

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from gravmag import eqlayer, filters, transforms, convolve
from gravmag.models import rectangular_prism as rp
from gravmag import plot_functions as plf
from gravmag import constants as cts

### Model

In [ ]:
# Create a model formed by a single prism
#model = np.array([[-500, 500, 0, 2000, 10, 1010]])
model = np.array([[-500, 500, 0, 2000, 150, 1010]])

density = np.array([2000]) # 2000 kg/m³ = 2 g/cm³

### Points were we compute the synthetic data

In [ ]:
# number of points along x and y
shape = (60, 60)

# minimum x, maximum x, minimum y and maximum y
area = [-5000, 5000, -4000, 6000]

# coordinates 
xp = np.linspace(area[0], area[1], shape[0])
yp = np.linspace(area[2], area[3], shape[1])
yp, xp = np.meshgrid(yp, xp)

yp = np.ravel(yp)
xp = np.ravel(xp)

# constant vertical coordinate of the simulated gravity data
z_data = -150

zp = np.zeros_like(yp) + z_data

coordinates = np.vstack([xp, yp, zp])

In [ ]:
# Grid spacing
dx = (area[1] - area[0])/(shape[0]-1)
dy = (area[3] - area[2])/(shape[1]-1)

In [ ]:
print(dx, dy)

### Synthetic gravitational field

In [ ]:
gz = rp.grav(coordinates=coordinates, prisms=model, density=density, field='z')

### Equivalent layer

Compute the physical-property distribution on the layer

In [ ]:
# layer z
z_layer = 2.5*(0.5*(dx+dy))

In [ ]:
# kernel matrix associated with the equivalent layer
G = eqlayer.kernel_matrix_monopoles(coordinates, z_layer, field='x')

In [ ]:
p_estimated = np.linalg.solve(G.T@G, G.T@gz)

In [ ]:
plt.figure(figsize=(7,5))

bounds = np.max(np.abs(p_estimated))

plt.title('Equivalent layer', fontsize=14)
plt.axis('scaled')
plt.contourf(
    yp.reshape(shape)*0.001, 
    xp.reshape(shape)*0.001, 
    p_estimated.reshape(shape), 
    cmap='RdBu_r',
    vmin = -bounds, vmax = bounds
)
plt.colorbar()
plf.model_boundaries(model, m2km=True)
plt.ylim(0.001*area[0], 0.001*area[1])
plt.xlim(0.001*area[2], 0.001*area[3])
plt.xlabel('y (km)', fontsize=12)
plt.ylabel('x (km)', fontsize=12)

plt.tight_layout()

plt.show()

### Gravitational field predicted by the equivalent layer

In [ ]:
# compute the predicted field and update the list
predicted_fields = [G@p_estimated]

### Plot the results

In [ ]:
true_fields = [gz]

bounds, diffs = plf.bounds_diffs(predicted_fields, true_fields)

fields = plf.fields_list(predicted_fields, true_fields, diffs)

titles = ['predicted gz (mGal)', 'true gz (mGal)', 'difference (mGal)']

windows = [i for i in range(1,4)]

plt.figure(figsize=(15,4))

for w, window in enumerate(windows):

    plt.subplot(1,3,window)
    plt.title(titles[w], fontsize=14)
    plt.axis('scaled')
    plt.contourf(
        yp.reshape(shape)*0.001, 
        xp.reshape(shape)*0.001, 
        fields[w].reshape(shape), 
        cmap='RdBu_r',
        vmin = -bounds[w], vmax = bounds[w]
    )
    plt.colorbar()
    plf.model_boundaries(model, m2km=True)
    plt.ylim(0.001*area[0], 0.001*area[1])
    plt.xlim(0.001*area[2], 0.001*area[3])
    plt.xlabel('y (km)', fontsize=12)
    plt.ylabel('x (km)', fontsize=12)

plt.tight_layout()

plt.show()